In [15]:
import pandas as pd
import requests
import os
import csv
import time
from datetime import datetime
from qbittorrentapi import Client
import sys

In [16]:
game_names = pd.read_csv('game_names.csv', sep=';', encoding='utf-8')
game_names.head()

,id,logo,name,current,2025peak,all time peak,compare
0,1.0,Obraz,Counter-Strike 2,"819,966","1,862,531","1,862,531",+
1,2.0,Obraz,Monster Hunter Wilds,"7,592","1,384,608","1,384,608",+
2,3.0,Obraz,PUBG: BATTLEGROUNDS,"287,841","1,347,327","3,257,248",+
3,4.0,Obraz,Dota 2,"331,439","961,289","1,295,114",+
4,5.0,Obraz,Battlefield™ 6,"31,88","747,44","747,44",+


In [17]:
game_names = game_names.drop(['compare', 'id'], axis=1)

In [18]:
matched_games = []
failed_games = []
for index, row in game_names.iterrows():
    game_name = row['name'].strip()
    url = "https://store.steampowered.com/api/storesearch/"
    params = {'term': game_name, 'l': 'english', 'cc': 'US'}
    response = requests.get(url, params=params)
    if response.status_code == 200:
        data = response.json()
        if data['total'] > 0:
            matched_games.append((game_name, data['items'][0]['id']))
        else:
            failed_games.append(game_name)
    else:
        print(f"Failed to fetch data for {game_name}. Status code: {response.status_code}")


In [19]:
failed_games

['Battlefield™ 6 Open Beta',
 'Mecha BREAK Demo',
 'Monster Hunter Wilds Beta test',
 'Source SDK Base 2007',
 'Grand Theft Auto V Legacy',
 'ARC Raiders Playtest',
 'REMATCH BETA TEST',
 '???????']

In [20]:
matched_games = [[item[0], item[1], ""] for item in matched_games]

for item in matched_games:
    app_id = item[1]
    url = f"https://store.steampowered.com/api/appdetails?appids={app_id}"
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        app_str = str(app_id)
        steam_name_raw = data[app_str]['data']['name']
        steam_name_clean = str(steam_name_raw).strip().lower()
        df_name_clean = str(item[0]).strip().lower()
        if steam_name_clean == df_name_clean:
            item[2] = "Match"
        else:
            item[2] = f'Mismatch (Steam name: {steam_name_raw})'

In [21]:
for game in matched_games:
    if game[2] != "Match":
        print(f"AppID {game[1]} - '{game[0]}' - {game[2]}")

AppID 3349410 - 'Path of Exile 2' - Mismatch (Steam name: Path of Exile 2 - Path of Exile 2 Early Access Supporter Pack)
AppID 1285190 - 'Borderlands® 4' - Mismatch (Steam name: Borderlands 4)
AppID 1172470 - 'Apex Legends' - Mismatch (Steam name: Apex Legends™)
AppID 1316910 - 'Banana' - Mismatch (Steam name: Super Monkey Ball Banana Mania)
AppID 2721690 - 'Call of Duty®' - Mismatch (Steam name: Cod Quest!)
AppID 3642000 - 'Spacewar' - Mismatch (Steam name: Space Warlord Baby Trading Simulator)
AppID 2395210 - 'skate.' - Mismatch (Steam name: Tony Hawk's™ Pro Skater™ 1 + 2)
AppID 2669320 - 'EA SPORTS FC 25' - Mismatch (Steam name: EA SPORTS FC™ 25)
AppID 2473420 - 'Football Manager 2024' - Mismatch (Steam name: Football Manager 2024 In-game Editor)
AppID 289070 - 'Sid Meier's Civilization VI' - Mismatch (Steam name: Sid Meier’s Civilization® VI)
AppID 3809270 - 'Euro Truck Simulator 2' - Mismatch (Steam name: Euro Truck Simulator 2 - Coaches)


In [22]:
for game in matched_games:
    if game[1] in [3349410, 2721690, 3642000, 2395210, 2473420, 3809270]:
        failed_games.append(game[0])
        matched_games.remove(game)

In [24]:
matched_df = pd.DataFrame(matched_games, columns=['name', 'appid', 'Steam_Verification'])
game_names = game_names.merge(matched_df, on='name', how='left')
game_names = game_names.dropna(subset=['appid'])
game_names['appid'] = game_names['appid'].astype(int)
game_names.head()


,logo,name,current,2025peak,all time peak,appid,Steam_Verification
0,Obraz,Counter-Strike 2,"819,966","1,862,531","1,862,531",730,Match
1,Obraz,Monster Hunter Wilds,"7,592","1,384,608","1,384,608",2246340,Match
2,Obraz,PUBG: BATTLEGROUNDS,"287,841","1,347,327","3,257,248",578080,Match
3,Obraz,Dota 2,"331,439","961,289","1,295,114",570,Match
4,Obraz,Battlefield™ 6,"31,88","747,44","747,44",2807960,Match


In [8]:
start_2025 = 1735689600
end_2025 = 1778845581
review_folder = '2025_reviews_2'
cursor = '*'

os.makedirs(review_folder, exist_ok=True)

def get_reviews(app_id):
    output_file = os.path.join(review_folder, f"reviews_{app_id}_2025.csv")
    with open(output_file, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["date", "voted_up", "votes_helpful", "text"])
            empty_page_strikes = 0

            while True:
                url = f"https://store.steampowered.com/appreviews/{app_id}?json=1"
                params = {
                    'filter': 'recent',
                    'language': 'english',
                    'num_per_page': 100,
                    'cursor': cursor,
                    'purchase_type': 'all',
                    'review_type': 'all'
                }
                try:
                    response = requests.get(url, params=params, timeout=10)
                    data = response.json()
                except Exception:
                    time.sleep(10)
                    continue

                
                if 'reviews' not in data or len(data['reviews']) == 0:
                    empty_page_strikes += 1
                    if empty_page_strikes >= 3:
                        break
                    time.sleep(3)
                    continue
                else:
                    empty_page_strikes = 0
            

                for review in data['reviews']:
                    timestamp = review['timestamp_created']
                    if start_2025 <= timestamp <= end_2025:
                        date_str = datetime.utcfromtimestamp(timestamp).strftime('%Y-%m-%d')
                        text = review['review'].replace('\n', ' ').replace('\r', '').strip()
                        if text:
                            writer.writerow([date_str, review['voted_up'], review['votes_up'], text])
                    elif timestamp < start_2025:
                        print("Done!")

In [ ]:
for idx, game in game_names.iterrows():
    print(f"Collecting reviews for {game['name']}...")
    get_reviews(game['appid'])

C:\Users\magda\AppData\Local\Temp\ipykernel_21824\3420348523.py:46: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  date_str = datetime.utcfromtimestamp(timestamp).strftime('%Y-%m-%d')


KeyboardInterrupt: 

In [17]:
def get_subreddits(game_name, app_id):
    headers = {'User-Agent': 'python:magisterka_scraper:v1.1 (by /u/little_programmer_1)'}
    banned_subs = {'gaming','games','pcgaming','videogames','askreddit','steam','funny','videos','test', 'furry', 'xbox', 'teenagers', 'Trophies', 'PS5'}
    min_subscribers = 5000

    search_term = game_name.replace(':', ' ').replace('®', '').replace('™', '').strip()

    url = "https://www.reddit.com/search.json"
    query_params = {'q': search_term, 'type': 'sr', 'limit': 5}

    max_retries = 5
    attempt = 0

    while attempt < max_retries:
        response = requests.get(url, headers=headers, params=query_params, timeout=10)

        if response.status_code == 429:
            print("[RATE LIMITED] retrying...")
            time.sleep(60)
            attempt += 1
            continue

        break

    game_subs = []

    try:
        data = response.json()

        for child in data.get('data', {}).get('children', []):
            sr_data = child['data']
            sub_name = sr_data['display_name']
            subscribers = sr_data.get('subscribers') or 0

            if sub_name.lower() not in banned_subs and subscribers >= min_subscribers:
                game_subs.append(sub_name)

    except Exception as e:
        print(f"Error: {e}")

    return game_subs

In [14]:
subreddits_list = []

for idx, game in game_names.iterrows():
    print(f"Searching subreddits for {game['name']}...")
    subs = get_subreddits(game['name'], game['appid'])

    subreddits_list.append({
        "appid": game["appid"],
        "game_name": game["name"],
        "subreddits": subs
    })

subreddits = pd.DataFrame(subreddits_list)

Searching subreddits for Counter-Strike 2...
Searching subreddits for Monster Hunter Wilds...
Searching subreddits for PUBG: BATTLEGROUNDS...
Searching subreddits for Dota 2...
Searching subreddits for Battlefield™ 6...
Searching subreddits for Marvel Rivals...
Searching subreddits for Hollow Knight: Silksong...
Searching subreddits for ARC Raiders...
Searching subreddits for Schedule I...
Searching subreddits for Path of Exile 2...
Searching subreddits for ELDEN RING NIGHTREIGN...
Searching subreddits for Borderlands® 4...
Searching subreddits for Escape from Duckov...
Searching subreddits for R.E.P.O....
Searching subreddits for Apex Legends...
Searching subreddits for Rust...
Searching subreddits for Split Fiction...
Searching subreddits for Kingdom Come: Deliverance II...
Searching subreddits for Where Winds Meet...
Searching subreddits for Delta Force...
Searching subreddits for NARAKA: BLADEPOINT...
Searching subreddits for Dispatch...
Searching subreddits for The Elder Scrolls I

In [16]:
subreddits.to_csv('game_subreddits_for_notebook.csv', index=False, encoding='utf-8')

In [ ]:
def get_torrents():
    print("Connecting to qBittorrent...")
    qbt_client = Client(host='localhost:8080')
    target_subs = []
    for sub_string in subreddits['subreddits'].dropna():
        individual_subs = sub_string.split(";")
        target_subs.extend([s.lower().strip() for s in individual_subs if s.strip()])
    target_subs = list(set(target_subs))

    torrents = qbt_client.torrents_info()
    reddit_torrent = next(t for t in torrents if "reddit" in t.name.lower(), None)

    if not reddit_torrent:
        print("Could not find the Reddit dump torrent.")
        sys.exit(1)

    files = qbt_client.torrents_files(torrent_hash=reddit_torrent.hash)
    file_ids_to_enable = []

    for f in files:
        filename_lower = f.name.lower()
        is_target = any(
            filename_lower.endswith(f"{sub}_submissions.zst") or 
            filename_lower.endswith(f"{sub}_comments.zst") or
            filename_lower == f"{sub}.zst"
            for sub in target_subs
        )

        if is_target:
            if f.priority == 0:
                file_ids_to_enable.append(f.index)
                print(f"  [QUEUED FOR DOWNLOAD] -> {f.name}")

    if file_ids_to_enable:
        qbt_client.torrents_file_priority(
            torrent_hash=reddit_torrent.hash, 
            file_ids=file_ids_to_enable, 
            priority=1
        )


In [ ]:
get_torrents()

In [25]:
game_names.to_csv('game_names_with_appids_to_notebook.csv', index=False, encoding='utf-8')